# Lesson 3 — Selecting & filtering
**Goal:** answer business questions by picking the rows and columns you need: `[]`, `loc`, `iloc`, boolean masks and `query`.

In [1]:
import pandas as pd
orders = pd.read_csv('data/orders.csv', parse_dates=['order_date'])
orders['revenue'] = orders['quantity'] * orders['unit_price'] * (1 - orders['discount_pct']/100)

## Columns

In [2]:
orders['branch'].head(3)                       # one column -> Series

0    Mlimani City
1        Kariakoo
2    Mlimani City
Name: branch, dtype: str

In [3]:
orders[['order_id', 'branch', 'revenue']].head(3)  # several columns -> DataFrame (double brackets)

,order_id,branch,revenue
0,O00001,Mlimani City,75000.0
1,O00002,Kariakoo,24000.0
2,O00003,Mlimani City,23750.0


## Rows by label (`loc`) and by position (`iloc`)

In [4]:
orders.iloc[0:3]          # first three rows by position

,order_id,order_date,customer_id,product_id,quantity,branch,channel,payment_method,unit_price,discount_pct,revenue
0,O00001,2025-01-01,C0027,P001,3,Mlimani City,In-store,Cash,25000,0,75000.0
1,O00002,2025-01-01,C0120,P012,2,Kariakoo,In-store,Card,12000,0,24000.0
2,O00003,2025-01-01,C0181,P001,1,Mlimani City,Online,Cash,25000,5,23750.0


In [5]:
orders.loc[5, ['order_id', 'product_id', 'revenue']]   # row label 5, chosen columns

order_id       O00006
product_id       P010
revenue       93500.0
Name: 5, dtype: object

## Boolean filters — the workhorse
**Question:** which orders were Online *and* above TSh 500,000?

In [6]:
big_online = orders[(orders['channel'] == 'Online') & (orders['revenue'] > 500_000)]
print(len(big_online), 'orders')
big_online[['order_id', 'order_date', 'product_id', 'revenue']].head()

37 orders


,order_id,order_date,product_id,revenue
7,O00008,2025-01-02,P008,1170000.0
96,O00097,2025-01-22,P008,1300000.0
134,O00135,2025-01-29,P008,552500.0
165,O00166,2025-02-03,P008,1300000.0
197,O00198,2025-02-10,P008,552500.0


Use `&` (and), `|` (or), `~` (not) — and wrap each condition in brackets.

In [7]:
# Orders from the two Dar es Salaam branches, paid by card
dar = orders[orders['branch'].isin(['Kariakoo', 'Mlimani City']) & (orders['payment_method'] == 'Card')]
dar.shape

(202, 11)

In [8]:
# December orders only
dec = orders[orders['order_date'].dt.month == 12]
print(len(dec), 'December orders, TSh', f"{dec['revenue'].sum():,.0f}")

368 December orders, TSh 45,434,700


## `query` — the same filter, easier to read

In [9]:
orders.query("channel == 'Online' and revenue > 500000").shape

(37, 11)

## Sorting & top-N

In [10]:
orders.sort_values('revenue', ascending=False).head(5)[['order_id', 'product_id', 'quantity', 'revenue']]

,order_id,product_id,quantity,revenue
2044,O02045,P008,2,1300000.0
260,O00261,P008,2,1300000.0
1997,O01998,P008,2,1300000.0
2001,O02002,P008,2,1300000.0
96,O00097,P008,2,1300000.0


In [11]:
orders.nlargest(3, 'revenue')[['order_id', 'branch', 'revenue']]

,order_id,branch,revenue
96,O00097,Mlimani City,1300000.0
165,O00166,Mlimani City,1300000.0
185,O00186,Mlimani City,1300000.0


## Key takeaways
- `df[col]`, `df[[cols]]`, `df.loc[rows, cols]` (labels), `df.iloc[rows, cols]` (positions).
- Filter with masks: `df[(cond1) & (cond2)]`; `isin` for lists; `query` for readability.

**Exercise:** find all Mobile Money orders at Arusha CBD with a discount. How many are there?